### Start SparkSession

In [59]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
import pandas as pd

# Самая простая инициализация
spark = SparkSession.builder \
    .appName("RetailAnalysis") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark готов")


Spark готов


In [60]:

import os

csv_path = "data/OnlineRetail.csv"

if os.path.exists(csv_path):
    print(f"CSV файл уже существует: {csv_path}")
    print("Можно сразу использовать в Spark!")
    
    # Проверяем
    file_size = os.path.getsize(csv_path) / (1024*1024)
    print(f"Размер файла: {file_size:.1f} MB")
    
else:
    print("Нужно установить openpyxl для конвертации Excel")
    print("Запусти: !pip install openpyxl")

CSV файл уже существует: data/OnlineRetail.csv
Можно сразу использовать в Spark!
Размер файла: 43.5 MB


In [61]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Инициализация Spark
spark = SparkSession.builder \
    .appName("RetailAnalysis") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# 2. Чтение CSV (у тебя уже есть!)
df = spark.read.csv("data/OnlineRetail.csv", 
                    header=True, 
                    inferSchema=True)

print(f"Загружено {df.count():,} строк")
print(f"Колонок: {len(df.columns)}")

print("\nПервые 5 строк:")
df.show(5)

# 3. Сохраняем для RFM
raw_df = df
print("Данные готовы для анализа!")

Загружено 541,909 строк
Колонок: 8

Первые 5 строк:
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity| InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/10 8:26|     2,55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/10 8:26|     3,39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/10 8:26|     2,75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/10 8:26|     3,39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/10 8:26|     3,39|     17850|United Kingdom|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
only showing top 5 rows

Данные готовы 

In [62]:

from pyspark.sql import functions as F

# 1. Очистка данных
print("\n1. Очистка данных...")
clean_df = raw_df.filter(
    (F.col("Quantity") > 0) &
    (F.col("UnitPrice") > 0) &
    (F.col("CustomerID").isNotNull())
)

print(f"   Было: {raw_df.count():,} строк")
print(f"   Стало: {clean_df.count():,} строк")

# 2. Сначала посмотрим формат дат в данных
print("\n2. Анализируем формат дат...")
print("   Примеры дат из файла:")
clean_df.select("InvoiceDate").limit(5).show(truncate=False)

# 3. Преобразование дат (правильный формат!)
print("\n3. Преобразование данных...")

# Попробуем разные форматы дат
try:
    # Формат 1: M/d/yy H:mm (год 10 вместо 2010)
    processed_df = clean_df.withColumn(
        "InvoiceDate",
        F.to_timestamp(F.col("InvoiceDate"), "M/d/yy H:mm")
    )
except Exception as e1:
    print(f"   Формат M/d/yy не сработал: {e1}")
    try:
        # Формат 2: d/M/yy H:mm (европейский)
        processed_df = clean_df.withColumn(
            "InvoiceDate",
            F.to_timestamp(F.col("InvoiceDate"), "d/M/yy H:mm")
        )
    except Exception as e2:
        print(f"   Формат d/M/yy не сработал: {e2}")
        # Формат 3: Просто как строка, потом вручную
        processed_df = clean_df.withColumn(
            "InvoiceDate",
            F.concat(
                F.substring(F.col("InvoiceDate"), 1, 6),
                F.lit("20"),
                F.substring(F.col("InvoiceDate"), 7, 2),
                F.substring(F.col("InvoiceDate"), 9, 100)
            )
        ).withColumn(
            "InvoiceDate",
            F.to_timestamp(F.col("InvoiceDate"), "M/d/yyyy H:mm")
        )

# Добавляем расчет цены
processed_df = processed_df.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)

print("   Пример преобразованных данных:")
processed_df.select("InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice", "TotalPrice").show(5)

# 4. RFM расчет
print("\n4. Расчет RFM метрик...")

# Находим максимальную дату в данных
max_date = processed_df.agg(F.max("InvoiceDate")).collect()[0][0]
print(f"   Последняя дата в данных: {max_date}")
print(f"   Используем для recency: {max_date}")

rfm_df = processed_df.groupBy("CustomerID").agg(
    # Recency: дни с последней покупки
    F.datediff(F.lit(max_date), F.max("InvoiceDate")).alias("recency_days"),
    
    # Frequency: количество уникальных транзакций
    F.countDistinct("InvoiceNo").alias("frequency"),
    
    # Monetary: общая сумма покупок
    F.sum("TotalPrice").alias("monetary"),
    
    # Дополнительные метрики
    F.count("*").alias("total_items"),
    F.min("InvoiceDate").alias("first_purchase"),
    F.max("InvoiceDate").alias("last_purchase")
).filter(F.col("monetary") > 0)

print(f"   Проанализировано клиентов: {rfm_df.count():,}")

print("\nТоп-10 клиентов по доходности:")
rfm_df.orderBy(F.col("monetary").desc()).show(10)

print("\nСтатистика RFM:")
rfm_df.select(
    F.avg("recency_days").alias("avg_recency"),
    F.avg("frequency").alias("avg_frequency"),
    F.avg("monetary").alias("avg_monetary"),
    F.sum("monetary").alias("total_revenue")
).show()

print("RFM анализ завершен!")


1. Очистка данных...
   Было: 541,909 строк
   Стало: 1,712 строк

2. Анализируем формат дат...
   Примеры дат из файла:
+-------------+
|InvoiceDate  |
+-------------+
|12/1/10 8:45 |
|12/1/10 10:29|
|12/1/10 11:27|
|12/1/10 13:04|
|12/1/10 14:05|
+-------------+


3. Преобразование данных...
   Пример преобразованных данных:
+---------+-------------------+--------+---------+----------+
|InvoiceNo|        InvoiceDate|Quantity|UnitPrice|TotalPrice|
+---------+-------------------+--------+---------+----------+
|   536370|2010-12-01 08:45:00|       3|       18|      54.0|
|   536392|2010-12-01 10:29:00|       1|      165|     165.0|
|   536403|2010-12-01 11:27:00|       1|       15|      15.0|
|   536527|2010-12-01 13:04:00|       1|       18|      18.0|
|   536540|2010-12-01 14:05:00|       1|       50|      50.0|
+---------+-------------------+--------+---------+----------+
only showing top 5 rows


4. Расчет RFM метрик...
   Последняя дата в данных: 2011-12-09 12:16:00
   Используем 

In [63]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. RFM Scoring (1-5) - исправленный
window_r = Window.orderBy(F.col("recency_days").asc())
window_f = Window.orderBy(F.col("frequency").desc())
window_m = Window.orderBy(F.col("monetary").desc())

# Определяем разные пути для сохранения
output_path_detailed = "rfm_results/rfm_results_detailed"
output_path_stats = "rfm_results/rfm_results_stats"

rfm_scored = rfm_df.withColumn("r_score", F.ntile(5).over(window_r)) \
                   .withColumn("f_score", F.ntile(5).over(window_f)) \
                   .withColumn("m_score", F.ntile(5).over(window_m))

# 2. Сегментация клиентов
rfm_segmented = rfm_scored.withColumn("segment",
    F.when((F.col("r_score") >= 4) & (F.col("f_score") >= 4), "Champions")
     .when((F.col("r_score") >= 4) & (F.col("f_score") >= 3), "Loyal")
     .when((F.col("r_score") >= 4), "New")
     .when((F.col("r_score") >= 3) & (F.col("f_score") >= 3), "Potential")
     .when((F.col("r_score") <= 2) & (F.col("f_score") >= 4), "At Risk")
     .when((F.col("r_score") <= 2) & (F.col("f_score") <= 2), "Lost")
     .otherwise("Others")
)



segment_stats = rfm_segmented.groupBy("segment").agg(
    F.count("*").alias("customer_count"),
    F.sum("monetary").alias("total_revenue"),
    F.round(F.avg("monetary"), 2).alias("avg_revenue_per_customer"),
    F.round(F.avg("recency_days"), 1).alias("avg_recency_days")
).orderBy(F.col("total_revenue").desc())

# Показываем результаты
segment_stats.show(truncate=False)

# 3. Сохраняем для дальнейшего использования
print("Сохраняю результаты анализа...")

# Сохраняем детальные данные по клиентам
rfm_segmented.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(output_path_detailed)

# Сохраняем агрегированную статистику по сегментам (в отдельную папку)
segment_stats.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(output_path_stats)

print("✅ Данные сохранены:")
print(f"   - {output_path_detailed} (детальные данные по клиентам)")
print(f"   - {output_path_stats} (агрегированная статистика по сегментам)")

# 4. Выводим итоговую статистику
total_customers = rfm_segmented.count()
total_revenue = rfm_segmented.agg(F.sum("monetary")).collect()[0][0]

25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 1

+---------+--------------+-------------+------------------------+----------------+
|segment  |customer_count|total_revenue|avg_revenue_per_customer|avg_recency_days|
+---------+--------------+-------------+------------------------+----------------+
|Lost     |144           |69749.0      |484.37                  |16.0            |
|Champions|196           |22775.0      |116.2                   |239.9           |
|Others   |164           |17529.0      |106.88                  |33.3            |
|New      |56            |14845.0      |265.09                  |173.5           |
|Potential|73            |5181.0       |70.97                   |64.3            |
+---------+--------------+-------------+------------------------+----------------+

Сохраняю результаты анализа...


25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 18:57:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/14 1

✅ Данные сохранены:
   - rfm_results/rfm_results_detailed (детальные данные по клиентам)
   - rfm_results/rfm_results_stats (агрегированная статистика по сегментам)
